In [52]:
from datasets import load_dataset

ds_reduced = load_dataset(
    "supermarine45/4be-dataset",
    data_files={
        "train_reduced": [
            "Option1/option1_nf_unsw_dos_as_ddos_reduced_schema/attack/train/*.csv",
            "Option1/option1_nf_unsw_dos_as_ddos_reduced_schema/normal/train/*.csv",
            "Option2/option2_nf_unsw_base_cse_native_ddos_reduced_schema/attack/train/*.csv",
            "Option2/option2_nf_unsw_base_cse_native_ddos_reduced_schema/normal/train/*.csv"
        ]
    }
)

ds_full = load_dataset(
    "supermarine45/4be-dataset",
    data_files={
        "train_full": [
            "Option1/option1_nf_unsw_dos_as_ddos/attack/train/*.csv",
            "Option1/option1_nf_unsw_dos_as_ddos/normal/train/*.csv",
            "Option2/option2_nf_unsw_base_cse_native_ddos/attack/train/*.csv",
            "Option2/option2_nf_unsw_base_cse_native_ddos/normal/train/*.csv"
        ]
    }
)

'(ProtocolError('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer')), '(Request ID: b4007918-5db1-4da3-a8d6-53f8c55f1992)')' thrown while requesting HEAD https://huggingface.co/datasets/supermarine45/4be-dataset/resolve/main/README.md
Retrying in 1s [Retry 1/5].


Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

In [53]:
ds_full["train_full"][1000]

{'FLOW_START_MILLISECONDS': 9000,
 'FLOW_END_MILLISECONDS': 10005,
 'IPV4_SRC_ADDR': '59.166.0.8',
 'L4_SRC_PORT': 12295,
 'IPV4_DST_ADDR': '149.171.126.4',
 'L4_DST_PORT': 80,
 'PROTOCOL': 6,
 'L7_PROTO': 7.0,
 'IN_BYTES': 1580,
 'IN_PKTS': 12,
 'OUT_BYTES': 10168,
 'OUT_PKTS': 18,
 'TCP_FLAGS': 27,
 'CLIENT_TCP_FLAGS': 27,
 'SERVER_TCP_FLAGS': 27,
 'FLOW_DURATION_MILLISECONDS': 1005,
 'DURATION_IN': 1005,
 'DURATION_OUT': 1004,
 'MIN_TTL': 31,
 'MAX_TTL': 32,
 'LONGEST_FLOW_PKT': 1500,
 'SHORTEST_FLOW_PKT': 52,
 'MIN_IP_PKT_LEN': 52,
 'MAX_IP_PKT_LEN': 1500,
 'SRC_TO_DST_SECOND_BYTES': 10.117412935323385,
 'DST_TO_SRC_SECOND_BYTES': 1.572139303482587,
 'RETRANSMITTED_IN_BYTES': 634,
 'RETRANSMITTED_IN_PKTS': 3,
 'RETRANSMITTED_OUT_BYTES': 4820,
 'RETRANSMITTED_OUT_PKTS': 4,
 'SRC_TO_DST_AVG_THROUGHPUT': 12564,
 'DST_TO_SRC_AVG_THROUGHPUT': 80858,
 'NUM_PKTS_UP_TO_128_BYTES': 18,
 'NUM_PKTS_128_TO_256_BYTES': 0,
 'NUM_PKTS_256_TO_512_BYTES': 6,
 'NUM_PKTS_512_TO_1024_BYTES': 0,
 'NUM_

In [54]:
print("df_reduced columns:")
print(df_reduced.columns.tolist())
print("\ndf_full columns:")
print(df_full.columns.tolist())

df_reduced columns:
['src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol', 'outbound_byte_ratio', 'duration', 'packets_per_second', 'bytes_per_second', 'inter_packet_arrival_mean', 'inter_packet_arrival_std', 'total_packets', 'total_bytes', 'packet_size_avg', 'packet_size_std', 'Label', 'Attack', 'scenario', 'split', 'dataset_id', 'row_in_window', 'is_seeded_ddos', 'burst_id', 'burst_phase', 'source_dataset']

df_full columns:
['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'IPV4_SRC_ADDR', 'L4_SRC_PORT', 'IPV4_DST_ADDR', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 'SRC_TO

In [55]:
import pandas as pd

# Convert to pandas dataframes
df_reduced = ds_reduced["train_reduced"].to_pandas()
df_full = ds_full["train_full"].to_pandas()

print(f"df_reduced shape: {df_reduced.shape}")
print(f"df_full shape: {df_full.shape}")

df_reduced shape: (3200000, 25)
df_full shape: (3200000, 64)


In [56]:
import pandas as pd
import numpy as np

# Feature policy for src_ip-window aggregated model
# Never use audit/leakage fields as model inputs.
audit_fields = [
    'Attack', 'Label', 'scenario', 'split', 'dataset_id',
    'burst_id', 'burst_phase', 'is_seeded_ddos', 'source_dataset', 'row_in_window'
]

# src_ip is the grouping key for aggregation, not a model input.
identity_fields = ['src_ip', 'dst_ip']

# Requested feature family mapped to available reduced-schema columns.
selected_feature_candidates = [
    'dst_port',                    # destination port behavior proxy
    'protocol',                    # protocol behavior
    'packets_per_second',          # packet rate
    'bytes_per_second',            # byte rate
    'duration',                    # duration
    'total_packets',               # traffic volume
    'total_bytes',                 # traffic volume
    'packet_size_avg',             # packet-size stats
    'packet_size_std',             # packet-size stats
    'outbound_byte_ratio',         # outbound ratio
    'inter_packet_arrival_mean',   # temporal behavior
    'inter_packet_arrival_std'     # temporal behavior
]

available_selected_features = [
    col for col in selected_feature_candidates if col in df_reduced.columns
]
missing_selected_features = [
    col for col in selected_feature_candidates if col not in df_reduced.columns
]

print('Selected features found:', available_selected_features)
if missing_selected_features:
    print('Selected features missing in df_reduced:', missing_selected_features)

# Build modeling frame from selected features only.
X_selected = df_reduced[available_selected_features].copy()

# Encode categorical selected features only (protocol is categorical behaviorally).
categorical_selected = [col for col in ['protocol'] if col in X_selected.columns]
if categorical_selected:
    X_selected = pd.get_dummies(X_selected, columns=categorical_selected, drop_first=True, dtype=float)

# Ensure numeric matrix and clean invalid rows.
X_selected = X_selected.apply(pd.to_numeric, errors='coerce')
X_selected = X_selected.astype(float)

# Binary target: Attack vs Benign
y_selected = (df_reduced['Attack'].astype(str).str.lower() != 'benign').astype(int)

mask_clean = ~(
    X_selected.isna().any(axis=1)
    | np.isinf(X_selected).any(axis=1)
    | y_selected.isna()
)

X_clean = X_selected.loc[mask_clean].reset_index(drop=True)
y_clean = y_selected.loc[mask_clean].reset_index(drop=True)

print(f'X_clean shape: {X_clean.shape}')
print('Class balance (0=Benign, 1=Attack):')
print(y_clean.value_counts(normalize=True).sort_index())

Selected features found: ['dst_port', 'protocol', 'packets_per_second', 'bytes_per_second', 'duration', 'total_packets', 'total_bytes', 'packet_size_avg', 'packet_size_std', 'outbound_byte_ratio', 'inter_packet_arrival_mean', 'inter_packet_arrival_std']
X_clean shape: (3200000, 17)
Class balance (0=Benign, 1=Attack):
Attack
0    0.990347
1    0.009653
Name: proportion, dtype: float64


In [57]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# VIF on selected feature set only
sample_size = min(10000, len(X_clean))
X_sample = X_clean.sample(n=sample_size, random_state=42)
X_vif = sm.add_constant(X_sample, has_constant='add')

print('Calculating VIF on selected features...')

vif_data = pd.DataFrame({
    'Feature': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})

vif_data = (
    vif_data[vif_data['Feature'] != 'const']
    .sort_values(by='VIF', ascending=False)
    .reset_index(drop=True)
)

print('\n' + '=' * 80)
print('VIF ON SELECTED FEATURES')
print('=' * 80)
print(vif_data.head(20))

high_vif_features = vif_data[vif_data['VIF'] > 10]['Feature'].tolist()
good_features = vif_data[vif_data['VIF'] <= 10]['Feature'].tolist()

print(f'\nHigh VIF features (>10): {len(high_vif_features)}')
print(high_vif_features[:20])
print(f'\nGood features (<=10): {len(good_features)}')
print(good_features[:20])

Calculating VIF on selected features...

VIF ON SELECTED FEATURES
                      Feature        VIF
0                 protocol_89        inf
1               protocol_ICMP        inf
2                protocol_TCP        inf
3                protocol_UDP        inf
4    inter_packet_arrival_std  35.074323
5   inter_packet_arrival_mean  34.083861
6             packet_size_avg  22.553782
7                 total_bytes  20.993962
8             packet_size_std  18.463156
9               total_packets  14.411518
10           bytes_per_second   4.294405
11                   duration   3.923173
12         packets_per_second   3.280072
13                   dst_port   1.593441
14        outbound_byte_ratio   1.457979
15                 protocol_2        NaN
16               protocol_211        NaN

High VIF features (>10): 10
['protocol_89', 'protocol_ICMP', 'protocol_TCP', 'protocol_UDP', 'inter_packet_arrival_std', 'inter_packet_arrival_mean', 'packet_size_avg', 'total_bytes', 'packet_siz

/Users/fagunawan/Library/Python/3.9/lib/python/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: divide by zero encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
/Users/fagunawan/Library/Python/3.9/lib/python/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
/Users/fagunawan/Library/Python/3.9/lib/python/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


In [58]:
import statsmodels.api as sm

# OLS on selected feature set only
X_ols = sm.add_constant(X_clean, has_constant='add')
ols_selected = sm.OLS(y_clean, X_ols).fit()

print('\n' + '=' * 80)
print('OLS RESULTS - SELECTED FEATURE SET')
print('=' * 80)
print(ols_selected.summary())


OLS RESULTS - SELECTED FEATURE SET
                            OLS Regression Results                            
Dep. Variable:                 Attack   R-squared:                       0.084
Model:                            OLS   Adj. R-squared:                  0.084
Method:                 Least Squares   F-statistic:                 1.729e+04
Date:                Sat, 09 May 2026   Prob (F-statistic):               0.00
Time:                        14:33:53   Log-Likelihood:             3.0403e+06
No. Observations:             3200000   AIC:                        -6.081e+06
Df Residuals:                 3199982   BIC:                        -6.080e+06
Df Model:                          17                                         
Covariance Type:            nonrobust                                         
                                coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------

In [59]:
# Significant features from OLS using selected feature set
print('\n' + '=' * 80)
print('SIGNIFICANT FEATURES (p < 0.05) - SELECTED SET')
print('=' * 80)

sig_features_selected = []
for feature in ols_selected.params.index:
    if feature == 'const':
        continue
    pvalue = ols_selected.pvalues[feature]
    if pvalue < 0.05:
        coef = ols_selected.params[feature]
        sig_features_selected.append({
            'feature': feature,
            'coefficient': coef,
            'p_value': pvalue,
            'abs_coef': abs(coef)
        })

sig_features_selected.sort(key=lambda x: x['abs_coef'], reverse=True)
print(f'Found {len(sig_features_selected)} significant features')
for i, feat in enumerate(sig_features_selected[:20], 1):
    print(f"{i:2d}. {feat['feature']:35s} | coef: {feat['coefficient']:12.6f} | p-value: {feat['p_value']:.2e}")

# Cross-check OLS significance with multicollinearity filter
sig_feature_names = {x['feature'] for x in sig_features_selected}
vif_ok_set = set(good_features)
final_candidate_features = sorted(sig_feature_names.intersection(vif_ok_set))

print('\n' + '=' * 80)
print('FINAL CANDIDATE FEATURES (OLS significant AND VIF <= 10)')
print('=' * 80)
print(final_candidate_features)
print(f'Total final candidate features: {len(final_candidate_features)}')

print('\nDropped by policy (audit/leakage fields):')
print(audit_fields + identity_fields)


SIGNIFICANT FEATURES (p < 0.05) - SELECTED SET
Found 15 significant features
 1. protocol_89                         | coef:    -1.051810 | p-value: 3.85e-244
 2. protocol_UDP                        | coef:    -0.991326 | p-value: 1.24e-221
 3. protocol_TCP                        | coef:    -0.984864 | p-value: 8.82e-219
 4. protocol_ICMP                       | coef:    -0.971460 | p-value: 4.25e-208
 5. outbound_byte_ratio                 | coef:    -0.023965 | p-value: 0.00e+00
 6. inter_packet_arrival_mean           | coef:     0.000437 | p-value: 0.00e+00
 7. duration                            | coef:     0.000353 | p-value: 5.51e-26
 8. inter_packet_arrival_std            | coef:    -0.000084 | p-value: 0.00e+00
 9. packet_size_avg                     | coef:    -0.000064 | p-value: 0.00e+00
10. packet_size_std                     | coef:     0.000040 | p-value: 0.00e+00
11. total_packets                       | coef:     0.000004 | p-value: 1.59e-241
12. packets_per_second    

In [60]:
print('final_candidate_features:', final_candidate_features)
print('num_final_features:', len(final_candidate_features))

final_candidate_features: ['bytes_per_second', 'dst_port', 'duration', 'outbound_byte_ratio', 'packets_per_second']
num_final_features: 5
